# Pre-Sales Transcript Analysis
## Comprehensive Data Analysis of Sales Conversation

This notebook analyzes a pre-sales conversation between Arjun (Pre-Sales) and Meera (Client) to extract insights, patterns, and actionable intelligence.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import re
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 1. Load and Parse Transcript Data

In [2]:
# Read the SRT transcript file
def parse_srt_file(file_path):
    """Parse SRT file and extract conversation data"""
    with open(file_path, 'r', encoding='utf-8') as file:
        content = file.read()
    
    # Split by double newlines to get individual subtitle blocks
    blocks = content.strip().split('\n\n')
    
    conversations = []
    for block in blocks:
        lines = block.strip().split('\n')
        if len(lines) >= 3:
            # Extract subtitle number
            subtitle_num = lines[0]
            
            # Extract timestamp
            timestamp_line = lines[1]
            time_match = re.match(r'(\d{2}:\d{2}:\d{2},\d{3}) --> (\d{2}:\d{2}:\d{2},\d{3})', timestamp_line)
            
            if time_match:
                start_time = time_match.group(1).replace(',', '.')
                end_time = time_match.group(2).replace(',', '.')
                
                # Extract speaker and text
                text = ' '.join(lines[2:])
                
                # Parse speaker and message
                speaker_match = re.match(r'^([^:]+):\s*(.+)', text)
                if speaker_match:
                    speaker = speaker_match.group(1)
                    message = speaker_match.group(2)
                else:
                    speaker = "Narrative"
                    message = text
                
                conversations.append({
                    'subtitle_num': int(subtitle_num),
                    'start_time': start_time,
                    'end_time': end_time,
                    'speaker': speaker,
                    'message': message
                })
    
    return pd.DataFrame(conversations)

# Load the transcript
df = parse_srt_file('transcript.srt')
print(f"Total conversation blocks: {len(df)}")
df.head(10)

Total conversation blocks: 67


,subtitle_num,start_time,end_time,speaker,message
0,1,00:00:00.000,00:00:18.000,Arjun (Pre-Sales),"Hi Meera, good afternoon. Can you hear me clea..."
1,2,00:00:18.500,00:00:28.000,Meera (Client),"Yes, loud and clear. Thanks for joining. I kno..."
2,3,00:00:28.500,00:00:42.000,Arjun,"Absolutely. I always say, the hardest part of ..."
3,4,00:00:42.500,00:00:57.000,Meera,Busy but good. We’re preparing for a board upd...
4,5,00:00:57.500,00:01:13.000,Arjun,Perfect. Then let’s make this session useful. ...
5,6,00:01:13.500,00:01:22.000,Meera,That’s exactly what we need. No generic slides...
6,7,00:01:22.500,00:01:44.000,Arjun,(laughs) Don’t worry. Let’s start with your pr...
7,8,00:01:44.500,00:02:10.000,Meera,Sure. We’re building a B2B SaaS platform focus...
8,9,00:02:10.500,00:02:32.000,Arjun,"Great. And when you say scaling, is it infrast..."
9,10,00:02:32.500,00:02:54.000,Meera,"Both. Today, we support ~1,000 shipments/day. ..."


## 2. Data Preprocessing and Feature Engineering

In [3]:
# Convert timestamps to seconds for analysis
def time_to_seconds(time_str):
    """Convert timestamp to seconds"""
    time_parts = time_str.replace('.', ':').split(':')
    hours = int(time_parts[0])
    minutes = int(time_parts[1])
    seconds = int(time_parts[2])
    milliseconds = int(time_parts[3])
    return hours * 3600 + minutes * 60 + seconds + milliseconds / 1000

df['start_seconds'] = df['start_time'].apply(time_to_seconds)
df['end_seconds'] = df['end_time'].apply(time_to_seconds)
df['duration'] = df['end_seconds'] - df['start_seconds']
df['word_count'] = df['message'].apply(lambda x: len(x.split()))
df['char_count'] = df['message'].apply(len)

# Clean speaker names
df['speaker_clean'] = df['speaker'].apply(lambda x: x.split('(')[0].strip() if '(' in x else x.strip())

# Identify speaker roles
df['role'] = df['speaker'].apply(lambda x: 
    'Pre-Sales' if 'Pre-Sales' in x or 'Arjun' in x 
    else 'Client' if 'Client' in x or 'Meera' in x 
    else 'Other'
)

print("Data preprocessing complete!")
df[['speaker_clean', 'role', 'duration', 'word_count']].head()

Data preprocessing complete!


,speaker_clean,role,duration,word_count
0,Arjun,Pre-Sales,18.0,9
1,Meera,Client,9.5,14
2,Arjun,Pre-Sales,13.5,24
3,Meera,Client,14.5,23
4,Arjun,Pre-Sales,15.5,38


## 3. Conversation Flow Analysis

In [4]:
# Speaker statistics
speaker_stats = df[df['role'] != 'Other'].groupby('role').agg({
    'subtitle_num': 'count',
    'duration': 'sum',
    'word_count': 'sum',
    'char_count': 'sum'
}).rename(columns={
    'subtitle_num': 'turns',
    'duration': 'total_duration',
    'word_count': 'total_words',
    'char_count': 'total_chars'
})

speaker_stats['avg_words_per_turn'] = speaker_stats['total_words'] / speaker_stats['turns']
speaker_stats['avg_duration_per_turn'] = speaker_stats['total_duration'] / speaker_stats['turns']

print("\nSpeaker Statistics:")
print(speaker_stats.round(2))


Speaker Statistics:
           turns  total_duration  total_words  total_chars  \
role                                                         
Client        33           680.5          503         3225   
Pre-Sales     33           787.0          692         4382   

           avg_words_per_turn  avg_duration_per_turn  
role                                                  
Client                  15.24                  20.62  
Pre-Sales               20.97                  23.85  


In [6]:
# Create conversation timeline visualization
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Conversation Timeline', 'Word Count Distribution'),
    row_heights=[0.6, 0.4]
)

# Timeline plot
for role in df['role'].unique():
    if role != 'Other':
        role_df = df[df['role'] == role]
        fig.add_trace(
            go.Scatter(
                x=role_df['start_seconds'],
                y=[role] * len(role_df),
                mode='markers',
                marker=dict(size=role_df['word_count']*0.5, opacity=0.6),
                name=role,
                text=role_df['message'].str[:50] + '...',
                hovertemplate='%{text}<br>Words: %{marker.size:.0f}<extra></extra>'
            ),
            row=1, col=1
        )

# Word count over time
fig.add_trace(
    go.Bar(
        x=df[df['role'] == 'Pre-Sales']['start_seconds'],
        y=df[df['role'] == 'Pre-Sales']['word_count'],
        name='Pre-Sales',
        marker_color='blue',
        opacity=0.7
    ),
    row=2, col=1
)

fig.add_trace(
    go.Bar(
        x=df[df['role'] == 'Client']['start_seconds'],
        y=-df[df['role'] == 'Client']['word_count'],
        name='Client',
        marker_color='red',
        opacity=0.7
    ),
    row=2, col=1
)

fig.update_xaxes(title_text="Time (seconds)", row=2, col=1)
fig.update_yaxes(title_text="Speaker", row=1, col=1)
fig.update_yaxes(title_text="Word Count", row=2, col=1)

fig.update_layout(
    height=600,
    title_text="Conversation Flow Analysis",
    showlegend=True
)

fig.write_html("plot1.html")

## 4. Key Business Metrics Extraction

In [8]:
# Extract key business metrics from conversation
business_metrics = {
    'Current Capacity': '1,000 shipments/day',
    'Target Capacity': '100,000 shipments/day',
    'Scaling Factor': '100x',
    'Team Size': '12 engineers',
    'Current Clients': '3 mid-sized logistics providers',
    'Timeline to Enterprise': '6 months',
    'MVP Launch': '6 months ago',
    'Sales Team': '2 salespeople + CEO',
    'Target Markets': 'India + Southeast Asia',
    'Compliance Target': 'SOC 2 Type I (this year), Type II (next year)'
}

metrics_df = pd.DataFrame(list(business_metrics.items()), columns=['Metric', 'Value'])

# Display as table
fig = go.Figure(data=[go.Table(
    header=dict(values=list(metrics_df.columns),
                fill_color='paleturquoise',
                align='left',
                font=dict(size=14)),
    cells=dict(values=[metrics_df.Metric, metrics_df.Value],
               fill_color='lavender',
               align='left',
               font=dict(size=12),
               height=30))
])

fig.update_layout(
    title='Key Business Metrics',
    height=400
)

fig.write_html("plot2.html")

## 5. Technology Stack Analysis

In [9]:
# Technology stack breakdown
tech_stack = {
    'Category': ['Cloud', 'Backend', 'Frontend', 'Database', 'Data Science', 
                  'Monitoring', 'DevOps', 'Container', 'Security'],
    'Current': ['AWS (EC2, S3)', 'Node.js + Express', 'React', 'RDS', 'Python', 
                'CloudWatch', 'Manual Scripts', 'None', 'Basic IAM'],
    'Maturity': [3, 4, 4, 3, 2, 2, 1, 0, 2],  # Scale of 0-5
    'Priority': ['High', 'Medium', 'Medium', 'High', 'High', 
                 'High', 'Critical', 'High', 'Critical']
}

tech_df = pd.DataFrame(tech_stack)

# Create radar chart for technology maturity
fig = go.Figure()

fig.add_trace(go.Scatterpolar(
    r=tech_df['Maturity'],
    theta=tech_df['Category'],
    fill='toself',
    name='Current Maturity',
    line_color='blue'
))

fig.add_trace(go.Scatterpolar(
    r=[5] * len(tech_df),
    theta=tech_df['Category'],
    fill='toself',
    name='Target Maturity',
    line_color='green',
    opacity=0.3
))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 5]
        )),
    showlegend=True,
    title="Technology Stack Maturity Assessment"
)

fig.write_html("plot3.html")

## 6. Requirements and Pain Points Analysis

In [10]:
# Extract pain points and requirements
pain_points = [
    {"Category": "Infrastructure", "Issue": "Manual deployments", "Impact": "High", "Priority": 5},
    {"Category": "Infrastructure", "Issue": "No Kubernetes", "Impact": "Medium", "Priority": 4},
    {"Category": "Scaling", "Issue": "1K to 100K shipments/day gap", "Impact": "Critical", "Priority": 5},
    {"Category": "Compliance", "Issue": "No SOC 2 readiness", "Impact": "Critical", "Priority": 5},
    {"Category": "Team", "Issue": "No DevOps engineer", "Impact": "High", "Priority": 5},
    {"Category": "Security", "Issue": "Basic security posture", "Impact": "High", "Priority": 4},
    {"Category": "Monitoring", "Issue": "No central logging", "Impact": "Medium", "Priority": 3},
    {"Category": "Integration", "Issue": "Limited API integrations", "Impact": "High", "Priority": 4},
    {"Category": "AI/ML", "Issue": "Early-stage models", "Impact": "Medium", "Priority": 3},
    {"Category": "Sales", "Issue": "Small sales team", "Impact": "Medium", "Priority": 3}
]

pain_df = pd.DataFrame(pain_points)

# Create priority heatmap
priority_matrix = pain_df.pivot_table(values='Priority', 
                                      index='Category', 
                                      columns='Impact', 
                                      fill_value=0)

fig = px.treemap(pain_df, 
                 path=['Impact', 'Category', 'Issue'], 
                 values='Priority',
                 color='Priority',
                 color_continuous_scale='RdYlBu_r',
                 title='Pain Points Priority Map')

fig.update_layout(height=500)
fig.write_html("plot4.html")

## 7. Competitive Analysis

In [11]:
# Competitive positioning analysis
competitors = pd.DataFrame({
    'Company': ['Client (Target)', 'Project44', 'FourKites'],
    'Price': [2, 5, 5],  # 1=Low, 5=High
    'Implementation Speed': [5, 2, 2],  # 1=Slow, 5=Fast
    'Customization': [5, 2, 2],  # 1=Low, 5=High
    'Enterprise Ready': [2, 5, 5],  # 1=Low, 5=High
    'Market Position': [1, 5, 4],  # 1=Low, 5=High
    'AI Capabilities': [3, 4, 4]  # 1=Low, 5=High
})

# Create competitive radar chart
categories = ['Price\n(Lower is Better)', 'Implementation\nSpeed', 'Customization', 
              'Enterprise\nReady', 'Market\nPosition', 'AI Capabilities']

fig = go.Figure()

for _, row in competitors.iterrows():
    fig.add_trace(go.Scatterpolar(
        r=row[1:].values,
        theta=categories,
        fill='toself',
        name=row['Company']
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 5]
        )),
    showlegend=True,
    title="Competitive Positioning Analysis"
)

fig.write_html("plot5.html")

## 8. Implementation Roadmap

In [12]:
# Create implementation timeline
roadmap_data = [
    dict(Task="Architecture Redesign", Start='2024-01-01', Finish='2024-02-15', Phase="Phase 1", Resource="Cloud Architect"),
    dict(Task="Compliance Roadmap", Start='2024-01-15', Finish='2024-02-28', Phase="Phase 1", Resource="Compliance Consultant"),
    dict(Task="DevOps Setup", Start='2024-02-01', Finish='2024-03-15', Phase="Phase 1", Resource="DevOps Engineer"),
    dict(Task="Scaling to 50K", Start='2024-03-01', Finish='2024-05-01', Phase="Phase 2", Resource="Full Team"),
    dict(Task="Integration Framework", Start='2024-03-15', Finish='2024-04-30', Phase="Phase 2", Resource="Integration Specialist"),
    dict(Task="SOC 2 Implementation", Start='2024-02-01', Finish='2024-05-31', Phase="Phase 1-2", Resource="Security Team"),
    dict(Task="AI Enhancements", Start='2024-05-01', Finish='2024-07-01', Phase="Phase 3", Resource="Data Science Team"),
    dict(Task="Enterprise Go-Live", Start='2024-06-15', Finish='2024-07-01', Phase="Phase 3", Resource="Full Team")
]

roadmap_df = pd.DataFrame(roadmap_data)

# Create Gantt chart
fig = px.timeline(roadmap_df, 
                  x_start="Start", 
                  x_end="Finish", 
                  y="Task", 
                  color="Phase",
                  hover_data=["Resource"],
                  title="Implementation Roadmap (6-Month Plan)")

fig.update_yaxes(autorange="reversed")
fig.update_layout(height=400)
fig.write_html("plot6.html")

## 9. Engagement Quality Metrics

In [13]:
# Calculate engagement metrics
total_duration = df['end_seconds'].max()
speaker_changes = (df['role'] != df['role'].shift()).sum()
avg_response_time = df['start_seconds'].diff().mean()

# Question analysis
questions = df[df['message'].str.contains('\?', na=False)]
questions_by_role = questions.groupby('role').size()

# Engagement score calculation
engagement_metrics = {
    'Total Duration (minutes)': round(total_duration / 60, 2),
    'Speaker Changes': speaker_changes,
    'Avg Response Time (seconds)': round(avg_response_time, 2),
    'Total Questions Asked': len(questions),
    'Questions by Pre-Sales': questions_by_role.get('Pre-Sales', 0),
    'Questions by Client': questions_by_role.get('Client', 0),
    'Conversation Balance': round(speaker_stats.loc['Pre-Sales', 'total_words'] / 
                                 speaker_stats.loc['Client', 'total_words'], 2)
}

# Calculate engagement score (0-100)
engagement_score = min(100, 
    (speaker_changes * 2) + 
    (100 - abs(50 - (questions_by_role.get('Pre-Sales', 0) / len(questions) * 100))) +
    (20 if engagement_metrics['Conversation Balance'] > 0.8 and 
          engagement_metrics['Conversation Balance'] < 1.2 else 0)
)

engagement_metrics['Engagement Score'] = round(engagement_score, 1)

# Display metrics
metrics_display = pd.DataFrame(list(engagement_metrics.items()), 
                               columns=['Metric', 'Value'])

fig = go.Figure(data=[go.Table(
    header=dict(values=list(metrics_display.columns),
                fill_color='lightblue',
                align='left',
                font=dict(size=14)),
    cells=dict(values=[metrics_display.Metric, metrics_display.Value],
               fill_color='white',
               align='left',
               font=dict(size=12),
               height=30))
])

fig.update_layout(
    title='Engagement Quality Metrics',
    height=350
)

fig.write_html("plot7.html")

## 10. Key Topics and Keywords Analysis

In [14]:
from collections import Counter
import string

# Extract all text
all_text = ' '.join(df['message'].str.lower())

# Define key business terms to track
key_terms = ['scaling', 'enterprise', 'compliance', 'soc 2', 'integration', 
             'aws', 'kubernetes', 'devops', 'ai', 'ml', 'api', 'security',
             'monitoring', 'dashboard', 'shipments', 'logistics', 'optimization',
             'budget', 'timeline', 'phase', 'architecture', 'infrastructure']

# Count occurrences
term_counts = {term: all_text.count(term) for term in key_terms}
term_counts = dict(sorted(term_counts.items(), key=lambda x: x[1], reverse=True)[:15])

# Create bar chart
fig = px.bar(x=list(term_counts.values()), 
             y=list(term_counts.keys()), 
             orientation='h',
             title='Key Topics Frequency',
             labels={'x': 'Frequency', 'y': 'Topic'},
             color=list(term_counts.values()),
             color_continuous_scale='Viridis')

fig.update_layout(height=500, showlegend=False)
fig.write_html("plot8.html")

## 11. Action Items and Next Steps

In [15]:
# Extract action items from the conversation
action_items = [
    {"Owner": "Client", "Action": "Share architecture diagrams", "Priority": "High", "Timeline": "Few days"},
    {"Owner": "Client", "Action": "Share DB sizes and current costs", "Priority": "High", "Timeline": "Few days"},
    {"Owner": "Client", "Action": "Share pipeline details", "Priority": "High", "Timeline": "Few days"},
    {"Owner": "Pre-Sales", "Action": "Prepare tailored proposal", "Priority": "High", "Timeline": "4-5 working days"},
    {"Owner": "Both", "Action": "Technical workshop", "Priority": "High", "Timeline": "Next week"},
    {"Owner": "Pre-Sales", "Action": "Share case studies", "Priority": "Medium", "Timeline": "With proposal"},
    {"Owner": "Pre-Sales", "Action": "Cost modeling by phases", "Priority": "High", "Timeline": "With proposal"},
    {"Owner": "Both", "Action": "Review proposal together", "Priority": "High", "Timeline": "After proposal"}
]

actions_df = pd.DataFrame(action_items)

# Create action items visualization
fig = px.sunburst(actions_df, 
                  path=['Owner', 'Priority', 'Action'], 
                  title='Action Items Breakdown',
                  color='Priority',
                  color_discrete_map={'High': '#FF4444', 'Medium': '#FFA500', 'Low': '#44FF44'})

fig.update_layout(height=500)
fig.write_html("plot9.html")

# Display action items table
print("\nAction Items Summary:")
for _, item in actions_df.iterrows():
    print(f"• [{item['Owner']}] {item['Action']} - Priority: {item['Priority']}, Timeline: {item['Timeline']}")


Action Items Summary:
• [Client] Share architecture diagrams - Priority: High, Timeline: Few days
• [Client] Share DB sizes and current costs - Priority: High, Timeline: Few days
• [Client] Share pipeline details - Priority: High, Timeline: Few days
• [Pre-Sales] Prepare tailored proposal - Priority: High, Timeline: 4-5 working days
• [Both] Technical workshop - Priority: High, Timeline: Next week
• [Pre-Sales] Share case studies - Priority: Medium, Timeline: With proposal
• [Pre-Sales] Cost modeling by phases - Priority: High, Timeline: With proposal
• [Both] Review proposal together - Priority: High, Timeline: After proposal


## 12. Executive Summary and Insights

In [16]:
# Generate executive summary
summary = f"""
# EXECUTIVE SUMMARY: Pre-Sales Call Analysis

## Call Overview
- **Duration**: {round(total_duration / 60, 1)} minutes
- **Participants**: Arjun (Pre-Sales) and Meera (Client)
- **Engagement Score**: {engagement_score:.1f}/100
- **Conversation Balance**: {engagement_metrics['Conversation Balance']} (Pre-Sales:Client word ratio)

## Client Profile
- **Product**: B2B SaaS platform for logistics optimization
- **Stage**: MVP launched 6 months ago
- **Current Scale**: 1,000 shipments/day → Target: 100,000 shipments/day
- **Team Size**: 12 engineers (no dedicated DevOps/Infrastructure)

## Key Business Requirements
1. **Scaling**: 100x capacity increase for enterprise clients
2. **Compliance**: SOC 2 Type I (immediate) and Type II (next year)
3. **Integration**: SAP, Oracle NetSuite, multiple carrier APIs
4. **Geographic Expansion**: India + Southeast Asia

## Critical Gaps Identified
- ❌ No DevOps engineer or infrastructure expertise
- ❌ Manual deployments with scripts
- ❌ No Kubernetes or container orchestration
- ❌ Basic security posture and monitoring
- ❌ Limited AI/ML capabilities

## Competitive Landscape
- **Main Competitors**: Project44, FourKites
- **Differentiation Strategy**: Faster, cheaper, more flexible for mid-market
- **Key Advantages**: Agility, cost-effectiveness, rapid deployment

## Proposed Solution Approach
- **Phase 1**: Architecture redesign + Compliance roadmap
- **Phase 2**: Scaling to 50K shipments/day
- **Phase 3**: AI/ML enhancements

## Deal Intelligence
- **Timeline Pressure**: First enterprise deal expected in 6 months
- **Budget Sensitivity**: ROI-focused, willing to invest where justified
- **Decision Process**: Direct engagement possible, RFP for multi-year
- **Trust Factor**: Previous disappointments with partners - wants transparency

## Recommended Actions
1. **Immediate**: Prepare highly customized proposal with clear ROI metrics
2. **Next Week**: Conduct technical workshop to validate assumptions
3. **Focus Areas**: DevOps maturity, compliance readiness, scaling architecture
4. **Differentiator**: Position as execution + consultative partner

## Win Probability Assessment: 75%
**Positive Indicators**:
- Clear pain points that match our capabilities
- Urgency driven by enterprise pipeline
- Budget available for right solution
- Good engagement and openness in discussion

**Risk Factors**:
- Past negative experiences with partners
- Budget constraints for comprehensive solution
- Tight timeline for enterprise readiness
"""

print(summary)


# EXECUTIVE SUMMARY: Pre-Sales Call Analysis

## Call Overview
- **Duration**: 25.3 minutes
- **Participants**: Arjun (Pre-Sales) and Meera (Client)
- **Engagement Score**: 100.0/100
- **Conversation Balance**: 1.38 (Pre-Sales:Client word ratio)

## Client Profile
- **Product**: B2B SaaS platform for logistics optimization
- **Stage**: MVP launched 6 months ago
- **Current Scale**: 1,000 shipments/day → Target: 100,000 shipments/day
- **Team Size**: 12 engineers (no dedicated DevOps/Infrastructure)

## Key Business Requirements
1. **Scaling**: 100x capacity increase for enterprise clients
2. **Compliance**: SOC 2 Type I (immediate) and Type II (next year)
3. **Integration**: SAP, Oracle NetSuite, multiple carrier APIs
4. **Geographic Expansion**: India + Southeast Asia

## Critical Gaps Identified
- ❌ No DevOps engineer or infrastructure expertise
- ❌ Manual deployments with scripts
- ❌ No Kubernetes or container orchestration
- ❌ Basic security posture and monitoring
- ❌ Limited AI/M

## 13. Export Results

In [17]:
# Save key analysis results
analysis_results = {
    'conversation_data': df.to_dict(),
    'speaker_statistics': speaker_stats.to_dict(),
    'business_metrics': business_metrics,
    'pain_points': pain_points,
    'action_items': action_items,
    'engagement_metrics': engagement_metrics,
    'term_frequency': term_counts
}

# Save to JSON for further analysis
import json

with open('pre_sales_analysis_results.json', 'w') as f:
    json.dump(analysis_results, f, indent=2, default=str)

print("Analysis complete! Results saved to 'pre_sales_analysis_results.json'")
print(f"\nTotal insights generated: {len(analysis_results)} categories")
print(f"Action items identified: {len(action_items)}")
print(f"Pain points documented: {len(pain_points)}")
print(f"Engagement score: {engagement_score:.1f}/100")

Analysis complete! Results saved to 'pre_sales_analysis_results.json'

Total insights generated: 7 categories
Action items identified: 8
Pain points documented: 10
Engagement score: 100.0/100
